# Setup

In [ ]:
!pip install -q -U "transformers==4.49.0" "accelerate>=1.3.0" "peft>=0.14" "bitsandbytes>=0.45"
!pip install -q -U "trl==0.13.0"
!pip install -q -U qwen-vl-utils bert-score rouge-score "torchao>=0.16"
!pip install -q "datasets<4.0.0"
!pip install -q gradio

In [ ]:
import os
import random
import logging

import numpy as np
import pandas as pd
import torch
import nltk

from datasets import load_dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else '-'}")

## HuggingFace Hub Login and Repos

In [ ]:
logging.getLogger("transformers").setLevel(logging.ERROR)
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
DATASET_REPO = "sdmikhalin/ecommerce-10k"
OUTPUT_REPO = "sdmikhalin/qwen25vl-3b-ecommerce-lora"

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

## Model, LoRA, Training parameters

In [ ]:
# Model
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
SAVE_STEPS = 50
LOGGING_STEPS = 10
OUTPUT_DIR = "/tmp/checkpoints"
FINAL_DIR = "/kaggle/working/qwen25vl_3b_ecommerce_lora"

# Prompt
SYSTEM_PROMPT = (
    "You are an e-commerce product description assistant for an online marketplace.\n"
    "Generate concise product titles in this format: [Brand] [Gender/Audience] [Color/Material] [Type].\n"
    "Keep it under 12 words. Be specific. Avoid filler words like 'this', 'a', 'an', 'image of'."
)

USER_PROMPTS = {
    "clothing": "Describe this clothing item as a marketplace product title. Include brand, gender, color, and type.",
    "electronics": "Describe this electronic device as a marketplace product title. Include brand, model, color, and key specifications.",
    "accessories": "Describe this accessory as a marketplace product title. Include brand, gender, color, material, and type.",
}
DEFAULT_USER_PROMPT = "Describe this product as a concise marketplace title."


def build_user_prompt(category):
    return USER_PROMPTS.get(category, DEFAULT_USER_PROMPT)

# Load Dataset from HF Hub

In [ ]:
ds = load_dataset(DATASET_REPO)
train_data = ds["train"]
test_data = ds["test"]

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Columns: {train_data.column_names}")
print(f"Train distribution: {pd.Series(train_data['category']).value_counts().to_dict()}")
print(f"Test distribution: {pd.Series(test_data['category']).value_counts().to_dict()}")

# Load Model and apply LORA

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# Collator

In [ ]:
class VLMCollator:
    def __init__(self, processor, system_prompt):
        self.processor = processor
        self.system_prompt = system_prompt

    def _build_messages(self, example):
        user_text = build_user_prompt(example["category"])
        return [
            {"role": "system", "content": [{"type": "text", "text": self.system_prompt}]},
            {"role": "user", "content": [
                {"type": "image", "image": example["image"]},
                {"type": "text", "text": user_text},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": example["text"]}]},
        ]

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            messages = self._build_messages(ex)
            text = self.processor.apply_chat_template(messages, tokenize=False)
            image_inputs, _ = process_vision_info(messages)
            texts.append(text)
            images.append(image_inputs)

        batch = self.processor(
            text=texts, images=images,
            return_tensors="pt", padding=True,
        )

        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        image_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
        if image_token_id is not None:
            labels[labels == image_token_id] = -100
        batch["labels"] = labels
        return batch


collator = VLMCollator(processor, SYSTEM_PROMPT)

# Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
    dataloader_num_workers=2,
    eval_strategy="no",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=collator,
)

trainer.train()

model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Adapter saved to {FINAL_DIR}")

# Publish Model to HF Hub

In [ ]:
model.push_to_hub(
    OUTPUT_REPO,
    private=False,
    commit_message="LoRA adapter for Qwen2.5-VL-3B: e-commerce 3 categories",
)
processor.push_to_hub(
    OUTPUT_REPO,
    private=False,
    commit_message="Processor config",
)
print(f"Published: https://huggingface.co/{OUTPUT_REPO}")